In [11]:
import networkx as nx
import numpy as np
from scipy.sparse import csc_matrix, diags

# 1. Caricamento Dati
mappatura_url = {}
lista_edges = []
with open('hollins.dat', 'r', encoding='utf-8') as f:
    intestazione = f.readline().split()
    numero_nodi = int(intestazione[0])
    numero_archi = int(intestazione[1])

    for i in range(numero_nodi):
        linea = f.readline().strip().split(' ', 1)
        node_id = int(linea[0])
        mappatura_url[node_id] = linea[1]

    for line in f:
        sorgente, destinazione = map(int, line.strip().split())
        lista_edges.append((sorgente, destinazione))

# Creazione Grafo
G = nx.DiGraph()
G.add_nodes_from(range(1, numero_nodi + 1))
G.add_edges_from(lista_edges)

# IMPORTANTE: Creiamo la lista ordinata dei nodi per garantire che l'indice i sia sempre ID i+1
nodi_ordinati = list(range(1, numero_nodi + 1))
N = numero_nodi

In [12]:
# Matrice di adiacenza con ordine prefissato
# nodelist garantisce che la riga 0 sia il nodo 1, riga 1 il nodo 2, etc.
A_nx = nx.adjacency_matrix(G, nodelist=nodi_ordinati)

# Calcolo Out-Degree
out_degrees = np.array([G.out_degree(n) for n in nodi_ordinati], dtype=float)

# Creazione Matrice Link (A_initial) - Colonna Stocastica
# Gestiamo i gradi zero per evitare divisioni per zero
with np.errstate(divide='ignore'):
    inv_out_degrees = 1.0 / out_degrees
inv_out_degrees[np.isinf(inv_out_degrees)] = 0.0

D_inv = diags(inv_out_degrees)
# A_initial ha colonne che sommano a 1 (tranne i dangling nodes che hanno colonne di 0)
A_initial = A_nx.T.dot(D_inv)

# Vettore h per i dangling nodes (1 se il nodo è dangling, 0 altrimenti)
h = (out_degrees == 0).astype(float).reshape(N, 1)

In [13]:
def PowerMethod_Stable(A_zero_cols, N, m, h, epsilon=1e-9, maxiter=500):
    d = 1.0 - m
    s = np.full((N, 1), 1.0 / N)  # Vettore di teletrasporto
    xk = s.copy() 

    for k in range(maxiter):
        xk_prev = xk.copy()

        # 1. Calcola la massa persa dai dangling nodes
        mass_lost = h.T.dot(xk_prev) # Scalare: quanta probabilità è "finita nel vuoto"

        # 2. Iterazione PageRank (Formula standard)
        # x = (1-m) * (A*x + s*mass_lost) + m*s
        xk_new = d * (A_zero_cols.dot(xk_prev) + s * mass_lost) + m * s

        # 3. Verifica Convergenza (Norma L1 è più precisa per probabilità)
        if np.linalg.norm(xk_new - xk_prev, ord=1) < epsilon:
            print(f"Convergenza raggiunta dopo {k+1} iterazioni.")
            return xk_new
        xk = xk_new

    return xk

# Esecuzione
m = 0.15
pagerank_vettore = PowerMethod_Stable(A_initial, N, m, h)

Convergenza raggiunta dopo 97 iterazioni.


In [14]:
# Trasformiamo il vettore in una lista piatta di score
pagerank_scores = pagerank_vettore.flatten()

classifica_completa = []
for i in range(N):
    node_id = nodi_ordinati[i] # Questo garantisce la corrispondenza corretta
    score = pagerank_scores[i]
    url = mappatura_url.get(node_id, "URL non trovato")
    classifica_completa.append((score, node_id, url))

# Ordinamento decrescente
classifica_completa.sort(key=lambda item: item[0], reverse=True)

print(f"\n🥇 RANKING REFACTORED (m={m})")
print("-" * 85)
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (score, node_id, url) in enumerate(classifica_completa[:10], start=1):
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")

# Risultati NetworkX (per verifica)
print("\n--- RISULTATI NETWORKX ---")
nx_scores_dict = nx.pagerank(G, alpha=0.85)
nx_scores = [nx_scores_dict[n] for n in sorted(G.nodes())]
print_top_ranking(nx_scores, mappatura_url)


🥇 RANKING REFACTORED (m=0.15)
-------------------------------------------------------------------------------------
Rank  Score           ID       URL
-------------------------------------------------------------------------------------
1     0.0198787507 2        http://www.hollins.edu/
2     0.0092876203 37       http://www.hollins.edu/admissions/visit/visit.htm
3     0.0086103930 38       http://www.hollins.edu/about/about_tour.htm
4     0.0080650307 61       http://www.hollins.edu/htdig/index.html
5     0.0080265649 52       http://www.hollins.edu/admissions/info-request/info-request.cfm
6     0.0071646430 43       http://www.hollins.edu/admissions/apply/apply.htm
7     0.0065827808 425      http://www.hollins.edu/academics/library/resources/web_linx.htm
8     0.0059892131 27       http://www.hollins.edu/admissions/admissions.htm
9     0.0055717361 28       http://www.hollins.edu/academics/academics.htm
10    0.0044524682 4023     http://www1.hollins.edu/faculty/saloweyca/clas%203